# Multilingual Sparse Baseline for BEA 2026

This notebook adds a PyTerrier-compatible multilingual sparse retrieval baseline using `opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1` and reports results in the same style as Table 1.

In [1]:
import os
import re
import unicodedata
from pathlib import Path
from typing import List, Tuple

import faiss
import numpy as np
import pandas as pd
import pyterrier as pt
import torch
from sentence_transformers import SentenceTransformer, SparseEncoder, util

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

_ws_re = re.compile(r"\s+")
_tags_re = re.compile(r"<[^>]+>")

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = _tags_re.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)
    s = s.lower()
    s = re.sub(r"[^0-9a-zA-ZäöüßÄÖÜẞ]+", " ", s)
    s = _ws_re.sub(" ", s).strip()
    return s

def safe_concat(parts):
    clean = []
    for p in parts:
        if p is None:
            continue
        p = str(p)
        if p.strip() in ("", "N/A", "nan"):
            continue
        clean.append(p)
    return " ".join(clean)

def build_doc_text(row: pd.Series) -> str:
    return safe_concat([row.get("question"), row.get("choices_processed")])

def ensure_run(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "qid" not in df.columns or "docno" not in df.columns:
        raise ValueError("Run must contain columns: qid, docno")
    df["qid"] = df["qid"].astype(str)
    df["docno"] = df["docno"].astype(str)
    if "score" not in df.columns:
        df["score"] = 0.0
    df["score"] = df["score"].astype(float)
    if "rank" not in df.columns:
        df = df.sort_values(["qid", "score"], ascending=[True, False])
        df["rank"] = df.groupby("qid").cumcount() + 1
    return df

if not pt.java.started():
    pt.java.init()

/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [2]:
PROJECT_ROOT = Path("/Users/user/Submissions/BEA-2026").resolve()
QB_PATH = PROJECT_ROOT / "qbank.csv"
QUERIES_PATH = PROJECT_ROOT / "queries.csv"
QRELS_PATH = PROJECT_ROOT / "qrels.tsv"

ART_DIR = PROJECT_ROOT / "artifacts" / "sparse_baseline"
ART_DIR.mkdir(parents=True, exist_ok=True)

DENSE_MODEL = "deutsche-telekom/gbert-large-paraphrase-cosine"
SPARSE_MODEL = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
DENSE_TOPK = 100
FINAL_TOPK = 50

qb = pd.read_csv(QB_PATH).fillna("N/A")
queries = pd.read_csv(QUERIES_PATH).fillna("")
qrels = pd.read_csv(QRELS_PATH, sep="\t").fillna(0)

qb["docno"] = qb["test_item_id"].astype(str)
qb["raw_text"] = qb.apply(build_doc_text, axis=1)
qb["text"] = qb["raw_text"].map(normalize_text)
corpus = qb[["docno", "text"]].copy()

topics = queries.rename(columns={"queries": "query"})[["qid", "query"]].copy()
topics["qid"] = topics["qid"].astype(str)
topics["query"] = topics["query"].astype(str).map(normalize_text)

qrels = qrels.rename(columns={"rel": "label"})[["qid", "docno", "label"]].copy()
qrels["qid"] = qrels["qid"].astype(str)
qrels["docno"] = qrels["docno"].astype(str)
qrels["label"] = qrels["label"].astype(int)

print("corpus:", corpus.shape)
print("topics:", topics.shape)
print("qrels:", qrels.shape)

corpus: (2812, 2)
topics: (15, 2)
qrels: (2755, 3)


In [3]:
class FaissDenseRetriever(pt.Transformer):
    def __init__(self, corpus_df: pd.DataFrame, model_name: str, topk: int, show_progress: bool = True):
        super().__init__()
        self.topk = int(topk)
        cdf = corpus_df[["docno", "text"]].copy()
        cdf["docno"] = cdf["docno"].astype(str)
        cdf["text"] = cdf["text"].astype(str)
        self.docnos = cdf["docno"].tolist()
        self.st = SentenceTransformer(model_name, device="cpu")
        xdoc = self.st.encode(
            cdf["text"].tolist(),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=bool(show_progress),
        ).astype("float32")
        self.index = faiss.IndexFlatIP(xdoc.shape[1])
        self.index.add(xdoc)

    def transform(self, topics_df: pd.DataFrame) -> pd.DataFrame:
        qids = topics_df["qid"].astype(str).tolist()
        qs = topics_df["query"].astype(str).tolist()
        q = self.st.encode(qs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
        scores, idxs = self.index.search(q, self.topk)
        rows = []
        for i, qid in enumerate(qids):
            for rank, (j, sc) in enumerate(zip(idxs[i], scores[i]), start=1):
                if j < 0:
                    continue
                rows.append({"qid": qid, "docno": self.docnos[j], "score": float(sc), "rank": int(rank)})
        return pd.DataFrame(rows)

class SparseEncoderRetriever(pt.Transformer):
    def __init__(self, corpus_df: pd.DataFrame, model_name: str, topk: int = 50, batch_size: int = 16):
        super().__init__()
        cdf = corpus_df[["docno", "text"]].copy()
        cdf["docno"] = cdf["docno"].astype(str)
        cdf["text"] = cdf["text"].astype(str)
        self.docnos = cdf["docno"].tolist()
        self.topk = int(topk)
        self.batch_size = int(batch_size)
        self.model = SparseEncoder(model_name, device="cpu")
        self.doc_emb = self.model.encode_document(
            cdf["text"].tolist(),
            batch_size=self.batch_size,
            convert_to_tensor=True,
            show_progress_bar=True,
        )

    def transform(self, topics_df: pd.DataFrame) -> pd.DataFrame:
        qids = topics_df["qid"].astype(str).tolist()
        qs = topics_df["query"].astype(str).tolist()
        query_emb = self.model.encode_query(
            qs,
            batch_size=self.batch_size,
            convert_to_tensor=True,
            show_progress_bar=True,
        )
        scores = util.dot_score(query_emb, self.doc_emb)
        rows = []
        for qi, qid in enumerate(qids):
            top = torch.topk(scores[qi], k=self.topk)
            for rank, (score, idx) in enumerate(zip(top.values.tolist(), top.indices.tolist()), start=1):
                rows.append({"qid": qid, "docno": self.docnos[int(idx)], "score": float(score), "rank": int(rank)})
        return pd.DataFrame(rows)

def cut_k(k: int) -> pt.Transformer:
    return pt.apply.generic(lambda df: df.groupby("qid", sort=False).head(int(k)))

def eval_at_k(pipelines: List[Tuple[str, pt.Transformer]], topics_df: pd.DataFrame, qrels_df: pd.DataFrame, k_eval: int) -> pd.DataFrame:
    metrics = [
        f"ndcg_cut.{k_eval}",
        "recip_rank",
        f"P.{k_eval}",
        f"recall.{k_eval}",
        f"map_cut.{k_eval}",
    ]
    df = pt.Experiment(
        [p for _, p in pipelines],
        topics_df,
        qrels_df,
        eval_metrics=metrics,
        names=[n for n, _ in pipelines],
        verbose=True,
        validate="ignore",
    )
    df = df.rename(columns={
        f"ndcg_cut.{k_eval}": f"nDCG@{k_eval}",
        "recip_rank": f"MRR@{k_eval}",
        f"P.{k_eval}": f"Prec@{k_eval}",
        f"recall.{k_eval}": f"Recall@{k_eval}",
        f"map_cut.{k_eval}": f"MAP@{k_eval}",
    }).copy()
    p = df[f"Prec@{k_eval}"].astype(float)
    r = df[f"Recall@{k_eval}"].astype(float)
    df[f"F1@{k_eval}"] = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
    cols = ["name", f"nDCG@{k_eval}", f"MRR@{k_eval}", f"Prec@{k_eval}", f"Recall@{k_eval}", f"F1@{k_eval}", f"MAP@{k_eval}"]
    return df[cols]

def to_table1_style(metrics_df: pd.DataFrame, ann_name: str, k_eval: int = 50) -> pd.DataFrame:
    df = metrics_df.copy()
    ann_ndcg = float(df.loc[df["name"] == ann_name, f"nDCG@{k_eval}"].iloc[0])
    df["Delta"] = df[f"nDCG@{k_eval}"].astype(float) - ann_ndcg
    df["Pct"] = np.where(ann_ndcg != 0, 100.0 * df["Delta"] / ann_ndcg, 0.0)
    out = pd.DataFrame({
        "Method": df["name"],
        "nDCG": df[f"nDCG@{k_eval}"].astype(float),
        "Delta": df["Delta"].astype(float),
        "%": df["Pct"].astype(float),
        "MRR": df[f"MRR@{k_eval}"].astype(float),
        "P": df[f"Prec@{k_eval}"].astype(float),
        "R": df[f"Recall@{k_eval}"].astype(float),
        "F1": df[f"F1@{k_eval}"].astype(float),
        "MAP": df[f"MAP@{k_eval}"].astype(float),
    })
    return out


In [4]:
dense100 = FaissDenseRetriever(corpus, model_name=DENSE_MODEL, topk=DENSE_TOPK)
sparse50 = SparseEncoderRetriever(corpus, model_name=SPARSE_MODEL, topk=FINAL_TOPK, batch_size=16)
cut50 = cut_k(FINAL_TOPK)

all_pipelines: List[Tuple[str, pt.Transformer]] = [
    ("ANN@100->@50", dense100 >> cut50),
    ("OpenSearch sparse encoder (multilingual)->@50", sparse50),
]

metrics_df = eval_at_k(all_pipelines, topics, qrels, k_eval=FINAL_TOPK)
metrics_df = metrics_df.sort_values(f"nDCG@{FINAL_TOPK}", ascending=False).reset_index(drop=True)
print(metrics_df.to_string(index=False))
metrics_df


Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Batches:   1%|          | 1/88 [00:05<07:20,  5.06s/it]

Batches:   2%|▏         | 2/88 [00:09<06:57,  4.85s/it]

Batches:   3%|▎         | 3/88 [00:13<06:15,  4.42s/it]

Batches:   5%|▍         | 4/88 [00:17<05:57,  4.25s/it]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 119.04it/s]


pt.Experiment: 100%|██████████| 2/2 [00:00<00:00,  3.81system/s]

                                         name  nDCG@50   MRR@50  Prec@50  Recall@50    F1@50   MAP@50
                                 ANN@100->@50 0.500596 0.794444 0.462667   0.411309 0.435479 0.273427
OpenSearch sparse encoder (multilingual)->@50 0.305359 0.713445 0.282667   0.180631 0.220413 0.112571


,name,nDCG@50,MRR@50,Prec@50,Recall@50,F1@50,MAP@50
0,ANN@100->@50,0.500596,0.794444,0.462667,0.411309,0.435479,0.273427
1,OpenSearch sparse encoder (multilingual)->@50,0.305359,0.713445,0.282667,0.180631,0.220413,0.112571


In [5]:
table1_df = to_table1_style(metrics_df, ann_name="ANN@100->@50", k_eval=FINAL_TOPK)
table1_df = table1_df.sort_values("nDCG", ascending=False).reset_index(drop=True)

for col in ["nDCG", "Delta", "%", "MRR", "P", "R", "F1", "MAP"]:
    table1_df[col] = table1_df[col].astype(float)

table1_df.to_csv(ART_DIR / "sparse_baseline_table1_style.csv", index=False)
metrics_df.to_csv(ART_DIR / "sparse_baseline_metrics_raw.csv", index=False)

print(table1_df.to_string(index=False))
table1_df


                                       Method     nDCG     Delta         %      MRR        P        R       F1      MAP
                                 ANN@100->@50 0.500596  0.000000   0.00000 0.794444 0.462667 0.411309 0.435479 0.273427
OpenSearch sparse encoder (multilingual)->@50 0.305359 -0.195237 -39.00085 0.713445 0.282667 0.180631 0.220413 0.112571


,Method,nDCG,Delta,%,MRR,P,R,F1,MAP
0,ANN@100->@50,0.500596,0.000000,0.00000,0.794444,0.462667,0.411309,0.435479,0.273427
1,OpenSearch sparse encoder (multilingual)->@50,0.305359,-0.195237,-39.00085,0.713445,0.282667,0.180631,0.220413,0.112571
